<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/11_3_MSA_based_AI_Agent_Ochestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 11-3] LangGraph 기반 MSA 에이전트 오케스트레이션 구현  

### 실습목표

- LangGraph의 기본 구조인 State, Node, Edge를 코드로 정의하고 연결할 수 있다.  

- 에이전트 간 직접 호출 대신 State 객체를 통해 데이터를 교환하는 MSA 방식의 협업을 구현한다.  

- 검토 결과에 따라 생성 단계로 되돌아가는 순환형(Cyclic) 자율 수정 루프를 구축한다.  

- 무한 루프를 방지하기 위한 종료 조건(Stop Condition) 설계의 중요성을 체험한다.  

1. 환경 준비 및 라이브러리 설치  

- LangGraph는 그래프 구조를 관리하기 위한 독립적인 라이브러리 설치가 필요합니다.  

In [ ]:
# 실습을 위한 라이브러리 설치
!pip install -q -U langgraph langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.48.0 which is incompatible.


In [ ]:
# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API 설정 완료


In [ ]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langgraph langchain-google-genai

import operator
from typing import Annotated, TypedDict, Union

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage

# Gemini API 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

2. [단계 1] 상태(State) 및 노드(Node) 정의  

- MSA의 공용 DB 역할을 하는 State를 정의하고, 독립적인 마이크로서비스 역할을 할 Node 함수를 작성합니다.  

In [ ]:
# (1) State 정의: 에이전트들이 공유할 '공용 메모리' [cite: 11-3-1]
class TeamState(TypedDict):
    task: str
    draft: str
    critique: str
    loop_count: Annotated[int, operator.add] # 반복 횟수 누적 계산

# (2) Node 구현: 독립적인 마이크로서비스 단위 [cite: 11-3-3]

# [Generator 노드] 초안을 작성하거나 수정함
def generator(state: TeamState):
    print(f"[*] Generator 실행 (반복 회차: {state['loop_count'] + 1})")

    prompt = f"주제: {state['task']}\n이전 피드백: {state['critique']}\n내용을 작성하거나 보완해주세요."
    response = llm.invoke(prompt)

    return {
        "draft": response.content,
        "loop_count": 1 # operator.add에 의해 기존 값에 +1됨
    }

# [Evaluator 노드] 결과물을 검토하고 피드백을 줌
def evaluator(state: TeamState):
    print("[*] Evaluator 실행 (품질 검토 중...)")

    # 50자 미만이면 무조건 반려하는 로직 (이론 교안 기준) [cite: 11-3-3]
    if len(state["draft"]) < 50:
        return {"critique": "내용이 너무 짧습니다. 50자 이상으로 더 상세히 작성하세요."}

    return {"critique": "PASS"}

3. [단계 2] 그래프 구성 및 순환 엣지(Edge) 연결  

- 정의한 노드들을 배치하고, 조건에 따라 '종료'할지 '재작성'할지 결정하는 Router를 설정합니다.  

In [ ]:
# (1) Router 구현: 조건부 분기 (Conditional Edge) [cite: 11-3-3]
def router(state: TeamState):
    # 종료 조건: PASS를 받았거나, 루프가 3회를 초과했을 때 (비용 폭탄 방지)
    if state["critique"] == "PASS" or state["loop_count"] >= 3:
        print("[!] 작업 완료 혹은 최대 반복 도달. 종료합니다.")
        return "finish"

    print("[!] 품질 미달. 재작성 루프로 진입합니다.")
    return "rewrite"

# (2) 그래프 설계 및 컴파일
workflow = StateGraph(TeamState)

# 노드 추가
workflow.add_node("generator", generator)
workflow.add_node("evaluator", evaluator)

# 엣지 연결
workflow.set_entry_point("generator") # 시작점 설정
workflow.add_edge("generator", "evaluator") # 생성 후 검토로 이동

# 조건부 엣지 추가: 검토 결과에 따라 분기 [cite: 11-3-2]
workflow.add_conditional_edges(
    "evaluator",
    router,
    {
        "finish": END,        # router가 finish 반환 시 종료
        "rewrite": "generator" # router가 rewrite 반환 시 다시 생성으로
    }
)

# 그래프 컴파일
app = workflow.compile()

4. [단계 3] 실행 및 결과 분석  

- 실제 에이전트가 스스로 성찰하며 결과물을 개선하는지 확인합니다.  

In [ ]:
# 초기 상태 설정 및 실행
initial_state = {
    "task": "AI 에이전트 오케스트레이션의 정의를 한 문장으로 설명해줘.",
    "draft": "",
    "critique": "",
    "loop_count": 0
}

# 실행
final_output = app.invoke(initial_state)

print("\n" + "="*50)
print(f"최종 결과물 (총 {final_output['loop_count']}회 반복):")
print(final_output['draft'])
print("="*50)

[*] Generator 실행 (반복 회차: 1)
[*] Evaluator 실행 (품질 검토 중...)
[!] 작업 완료 혹은 최대 반복 도달. 종료합니다.

최종 결과물 (총 1회 반복):
AI 에이전트 오케스트레이션은 복수의 독립적인 AI 에이전트들이 복잡한 공동의 목표를 달성하기 위해 상호 협력하고 조정되도록 관리하는 체계입니다.


**실습 분석**  

- MSA적 관점: Generator와 Evaluator는 서로의 내부 로직을 모르며 State만 주고받음  

- 자율 수정 루프: Evaluator의 피드백 내용에 따라 Generator가 답변을 보강함  

- 안정성 장치: loop_count 조건을 통해 무한 루프와 비용 과다 발생을 차단함  

**학습 점검**  

- Q1. LangGraph에서 Annotated[int, operator.add]가 MSA 설계에서 왜 유용한가요?  

    - A: 여러 노드가 동시에 혹은 반복적으로 상태를 업데이트할 때, 값을 덮어쓰지 않고 논리적으로 합산(누적)하여 상태를 관리할 수 있기 때문입니다. [cite: 11-3-1]  

- Q2. 자기 성찰 루프에서 'Router'가 담당하는 가장 중요한 시스템적 역할은?  

    - A: 비즈니스 요구사항(품질 기준) 충족 여부를 판단하여 워크플로우를 **종료(END)**하거나 재수행하도록 경로를 제어하는 오케스트레이터 역할을 수행합니다. [cite: 11-3-2]  

**학습정리**  

- LangGraph를 활용하면 에이전트를 독립적인 마이크로서비스로 격리하면서도 유기적인 상태 기반 협업이 가능합니다.  

- 자기 성찰 루프는 단순한 반복이 아니라, 데이터에 기반하여 품질을 스스로 높이는 지능형 오케스트레이션의 핵심입니다.  